# 03 — Predict & Visualise Channel Unmixing

Runs MicroSplit prediction on the demo dataset and visualises the unmixing results.

**Run notebooks 00–02 first.**

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
import torch

sys.path.insert(0, '../../../src')
from microsplit_reproducibility.workflows.cellpainting import (
    find_checkpoint,
    load_model_and_stats,
    predict_fov,
    compute_metrics,
    save_fov_predictions,
)

DEMO_DIR    = Path('./cpg0016_demo')
DATASET_DIR = DEMO_DIR / 'dataset'
PRED_DIR    = DEMO_DIR / 'predictions'
CHANNELS    = ['DNA', 'RNA', 'ER', 'AGP', 'Mito']

if not DATASET_DIR.exists():
    raise FileNotFoundError(f'{DATASET_DIR} not found — run earlier notebooks first.')

PRED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
checkpoint = find_checkpoint(str(DATASET_DIR))
model, stats = load_model_and_stats(
    training_dir=str(DATASET_DIR),
    checkpoint_path=checkpoint,
    channel_names=CHANNELS,
)
print(f'Checkpoint: {Path(checkpoint).name}  |  device: {next(model.parameters()).device}')

In [ ]:
image_id = 0
channel_images = {
    ch: tifffile.imread(str(DATASET_DIR / ch / f'{image_id:06d}.tiff'))
    for ch in CHANNELS
}

mmse_pred, posterior_samples = predict_fov(
    model=model,
    channel_images=channel_images,
    stats=stats,
    channel_names=CHANNELS,
    image_size=64,
    grid_size=8,
    multiscale_lowres_count=3,
    mmse_count=10,
    num_posterior_samples=1,
    posterior_seeds=(42,),
    batch_size=16,
)

In [ ]:
image_id = 0
channel_images = {
    ch: tifffile.imread(str(DATASET_DIR / ch / f'{image_id:06d}.tiff'))
    for ch in CHANNELS
}

# grid_size=64 gives ~289 tiles per 1080×1080 image (fast on CPU).
# Increase to grid_size=8 with mmse_count=50 for publication-quality predictions.
on_gpu = torch.cuda.is_available()
mmse_pred, posterior_samples = predict_fov(
    model=model,
    channel_images=channel_images,
    stats=stats,
    channel_names=CHANNELS,
    image_size=64,
    grid_size=32 if on_gpu else 64,
    multiscale_lowres_count=3,
    mmse_count=10 if on_gpu else 5,
    num_posterior_samples=1,
    posterior_seeds=(42,),
    batch_size=32,
)

In [ ]:
metrics = compute_metrics(channel_images, mmse_pred, CHANNELS)
df = pd.DataFrame([
    {'channel': ch, 'PSNR': metrics[f'psnr_{ch}'], 'SSIM': metrics[f'ssim_{ch}']}
    for ch in CHANNELS
])
print(df.to_string(index=False))
print(f"\nMean PSNR: {df['PSNR'].mean():.2f} dB   Mean SSIM: {df['SSIM'].mean():.4f}")

save_fov_predictions(
    output_dir=str(PRED_DIR),
    well='demo',
    site=1,
    mmse_prediction=mmse_pred,
    posterior_samples=posterior_samples,
    channel_names=CHANNELS,
    save_uint16=True,
)
print(f'Predictions saved → {PRED_DIR}')